In [21]:
import sys, os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/t1dbg'
except ImportError:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

### Import librerie


In [22]:
import os
import pandas as pd
import numpy as np
import json
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    mean_absolute_percentage_error,
)
import tensorflow as tf
import keras
from keras import layers

### Data Utilities


In [23]:
# Costanti
HYPO = 70.0
HYPER = 180.0
L_BOUND = 40.0
U_BOUND = 400.0

In [24]:
def load_splits(splits_dir="data/split_sets"):
    """Carica gli split e i metadati dai relativi file.
    Args:
        splits_dir (str): Directory contenente gli split set (default: 'data/split_sets')
    Returns:
        tuple: (train_set, val_set, test_set, X_cols, y_cols)
    """
    datasets = []
    for name in ["train", "val", "test"]:
        df = pd.read_parquet(f"{splits_dir}/{name}_set.parquet")
        datasets.append(df)

    with open(f"{splits_dir}/metadata.json", "r") as f:
        metadata = json.load(f)

    return tuple(datasets + [metadata["X_cols"], metadata["y_cols"]])

In [25]:
def rescale_data(df, rescale_cols):
    """Effettua il rescale dei dati back al loro intervallo originario.
    Args:
        df (pd.DataFrame): DataFrame con i dati da riscalare
        rescale_cols (list): List dei nomi delle colonne da riscalare
    Returns:
        pd.DataFrame: Dataframe con le colonne scelte riscalate
    """
    df = df.copy()
    for col in rescale_cols:
        df[col] = ((df[col] + 1) * (U_BOUND - L_BOUND) / 2) + L_BOUND
    return df

In [26]:
def calculate_metrics(df):
    """Calcola le metriche per ciascun paziente in un sottoinsieme.
    Args:
        df (pd.DataFrame): Dataframe con le colonne 'Patient_ID', 'target', e 'y_pred'
    Returns:
        tuple: (samples, maes, mapes, rmses) - numero di samples e lista delle metriche ottenute
    """
    samples = 0
    maes, mapes, rmses = [], [], []

    for patient_id in df["Patient_ID"].unique():
        patient_data = df[df["Patient_ID"] == patient_id]
        if patient_data.empty:
            continue

        samples += len(patient_data)
        maes.append(mean_absolute_error(patient_data["target"], patient_data["y_pred"]))
        mapes.append(
            mean_absolute_percentage_error(
                patient_data["target"], patient_data["y_pred"]
            )
            * 100
        )
        rmses.append(
            root_mean_squared_error(patient_data["target"], patient_data["y_pred"])
        )

    return samples, maes, mapes, rmses

In [27]:
def print_results(df):
    """Stampa i risultati delle valutazioni cumulative e per condizione glicemica.
    Args:
        df (pd.DataFrame): DataFrame con valori inferiti e di riferimento
    """

    def print_metrics(title, samples, maes, mapes, rmses):
        """Stampa le metriche formattate per una specifica condizione."""
        if title != "Cumulative":
            print("~" * 10)
        print(title)
        print(f"Samples: {samples}")
        if maes:  # Stampa solo se abbiamo dei dati
            print(f"MAE: {np.mean(maes):.2f}({np.std(maes):.2f})")
            print(f"MAPE: {np.mean(mapes):.2f}({np.std(mapes):.2f})")
            print(f"RMSE: {np.mean(rmses):.2f}({np.std(rmses):.2f})")

    # Overall results
    samples, maes, mapes, rmses = calculate_metrics(df)
    print_metrics("Cumulative", samples, maes, mapes, rmses)

    # Results by condition
    for condition in ["Normal", "Hyper", "Hypo"]:
        condition_df = df[df["bgClass"] == condition]
        samples, maes, mapes, rmses = calculate_metrics(condition_df)
        print_metrics(condition, samples, maes, mapes, rmses)

### Dnn Utilities


In [28]:
def prepare_data(train_set, val_set, test_set, X_cols, y_cols, model_type):
    """Prepara i dati per il training"""

    def _ensure_1d(array):
        """Assicurati che l'array sia unidimensionale"""
        return array.flatten() if array.ndim > 1 else array

    def _reshape_for_rnn(X):
        """Esegui il reshape de i dati per i modelli RNN"""
        return X.reshape(X.shape[0], X.shape[1], 1)

    if model_type not in ["mlp", "lstm", "gru"]:
        raise ValueError(f"Model type must be one of {['mlp', 'lstm', 'gru']}")

    # Extract and convert data in one step
    datasets = [train_set, val_set, test_set]
    X_arrays = [df[X_cols].values.astype(np.float32) for df in datasets]
    y_arrays = [df[y_cols].values.astype(np.float32) for df in datasets]

    # Ensure targets are 1D
    y_arrays = [_ensure_1d(y) for y in y_arrays]

    # Reshape for RNN models if needed
    if model_type in ["lstm", "gru"]:
        X_arrays = [_reshape_for_rnn(X) for X in X_arrays]

    # Print data information
    shapes = [
        (X_arrays[0], y_arrays[0], "train"),
        (X_arrays[1], y_arrays[1], "val"),
        (X_arrays[2], y_arrays[2], "test"),
    ]
    print("Data shapes:")
    for X, y, name in shapes:
        print(f"X_{name}: {X.shape}, y_{name}: {y.shape}")

    return tuple(X_arrays + y_arrays)

In [29]:
def create_mlp_model():
    """Costruisci il modello MLP"""
    return keras.Sequential(
        [
            layers.Input(shape=(8,)),
            layers.Dense(256, activation="tanh"),
            layers.Dropout(0.2),
            layers.Dense(256, activation="tanh"),
            layers.Dropout(0.2),
            layers.Dense(1),
        ],
        name="MLP_Model",
    )

In [30]:
def create_lstm_model():
    """Costruisci il modello LSTM"""
    return keras.Sequential(
        [
            layers.Input(shape=(8, 1)),
            layers.LSTM(75, return_sequences=True),
            layers.Dropout(0.2),
            layers.LSTM(75, return_sequences=False),
            layers.Dense(1),
        ],
        name="LSTM_Model",
    )

In [31]:
def create_gru_model():
    """Costruisci il modello GRU"""
    return keras.Sequential(
        [
            layers.Input(shape=(8, 1)),
            layers.GRU(86, return_sequences=True),
            layers.Dropout(0.2),
            layers.GRU(86, return_sequences=False),
            layers.Dense(1),
        ],
        name="GRU_Model",
    )

In [32]:
def create_model(model_type):
    """Costruisci il modello specificato"""
    if model_type not in ["mlp", "lstm", "gru"]:
        raise ValueError(f"Model type must be one of {['mlp', 'lstm', 'gru']}")

    if model_type == "mlp":
        return create_mlp_model()
    elif model_type == "lstm":
        return create_lstm_model()
    elif model_type == "gru":
        return create_gru_model()

In [33]:
def train_model(
    model,
    X_train,
    y_train,
    X_val,
    y_val,
    epochs=100,
    batch_size=4096,
    initial_learning_rate=0.01,
    models_path="models",
    exp_name="model",
):
    """Esegui il training per un modello specificato"""

    def create_callbacks(**kwargs):
        """Crea i callbacks per il training con i valori di default"""
        defaults = {
            "early_stopping_patience": 5,
            "early_stopping_min_delta": 1e-5,
            "lr_scheduler": True,
            "lr_reduction_factor": 0.1,
            "lr_scheduler_patience": 3,
            "min_learning_rate": 1e-6,
            "monitor": "val_loss",
            "verbose": 1,
        }

        config = {**defaults, **kwargs}

        callbacks = [
            keras.callbacks.EarlyStopping(
                monitor=config["monitor"],
                patience=config["early_stopping_patience"],
                min_delta=config["early_stopping_min_delta"],
                verbose=config["verbose"],
                restore_best_weights=True,
            )
        ]

        if config["lr_scheduler"]:
            callbacks.append(
                keras.callbacks.ReduceLROnPlateau(
                    monitor=config["monitor"],
                    factor=config["lr_reduction_factor"],
                    patience=config["lr_scheduler_patience"],
                    min_lr=config["min_learning_rate"],
                    verbose=config["verbose"],
                )
            )

        return callbacks

    # Compile model
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=initial_learning_rate),
        loss=keras.losses.Huber(),
        metrics=["mae"],
        steps_per_execution=256,
    )

    # Create callbacks
    callbacks = create_callbacks()

    # Train model
    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=1,
    )

    # Salva i pesi del miglior modello alla fine del training
    # (EarlyStopping con restore_best_weights=True già ripristina i migliori pesi)
    model_save_path = f"{models_path}/{exp_name}.weights.h5"
    os.makedirs(os.path.dirname(model_save_path), exist_ok=True)
    model.save_weights(model_save_path)
    print(f"Best model weights saved to: {model_save_path}")

    return model, history

In [34]:
def predict_in_batches(model, data, model_type, batch_size=256):
    """Esegui le predizione ed effettua il reshaping automatico dei dati"""
    if model_type not in ["mlp", "lstm", "gru"]:
        raise ValueError(f"Model type must be one of {['mlp', 'lstm', 'gru']}")

    def _reshape_for_rnn(X):
        """Esegui il reshape de i dati per i modelli RNN"""
        return X.reshape(X.shape[0], X.shape[1], 1)

    # Prepare data based on model type
    if model_type in ["lstm", "gru"]:
        data_reshaped = _reshape_for_rnn(data.values)
    else:
        data_reshaped = data.values

    return model.predict(data_reshaped, batch_size=batch_size, verbose=0)

In [35]:
def print_model_summary(model):
    """Stampa l'architettura del modello ed il conteggio dei parametri"""
    print("\nModel Architecture:")
    model.summary()
    print(f"\nTotal parameters: {model.count_params():,}")

### Training Dnn Utilities


In [36]:
def train(X_train, y_train, X_val, y_val, args, seed):
    """Esegui il training nella main pipeline"""

    def set_seeds(seed):
        """Setta i random seed per riproducibilità in TensorFlow/Keras."""
        # Set NumPy seed
        np.random.seed(seed)
        # Set TensorFlow seeds
        tf.keras.backend.clear_session()
        tf.random.set_seed(seed)
        tf.keras.utils.set_random_seed(seed)

    set_seeds(seed)

    # Costruisci e addestra il modello
    print(f"\nCreating {args.exp_name.upper()} model with seed {seed}...")
    model = create_model(args.exp_name)
    if seed == args.seed:  # Stampa la summary solo per la prima run
        print_model_summary(model)

    print(f"\nTraining {args.exp_name.upper()} model with seed {seed}...")
    model, history = train_model(
        model,
        X_train,
        y_train,
        X_val,
        y_val,
        args.epochs,
        args.batch_size,
        args.lr,
        args.models_path,
        f"{args.exp_name}_seed_{seed}",
    )

    return model, history

In [37]:
def evaluate_model(model, model_type, eval_set, X_cols, y_cols):
    """Valuta il modello e restituisci i risultati"""
    # Valuta il modello sul validation set
    print(f"\nEvaluating model...")
    eval_set = eval_set.copy()  # Non modificare il set originale
    eval_set["y_pred"] = predict_in_batches(model, eval_set[X_cols], model_type)
    eval_set = eval_set.rename(columns={y_cols[-1]: "target"})
    eval_set = rescale_data(eval_set, ["target", "y_pred"])

    # Selezione le colonne da mostrare nei risultati
    output_columns = ["Timestamp", "Patient_ID", "bgClass", "target", "y_pred"]
    results = eval_set[output_columns]

    return results

In [38]:
def calculate_mae(results):
    """Calcola il MAE dei pazienti dal DataFrame risultante"""
    patient_maes = []
    for patient_id in results["Patient_ID"].unique():
        patient_data = results[results["Patient_ID"] == patient_id]
        mae = np.mean(np.abs(patient_data["target"] - patient_data["y_pred"]))
        patient_maes.append(mae)
    return np.mean(patient_maes)

### Main pipeline


In [39]:
# Configura il training
class Args:
    def __init__(self):
        self.output_path = "outputs/val_set"
        self.models_path = "models/val_set"
        self.exp_name = "mlp"  # Cambia qui: "mlp", "lstm", "gru"
        self.seed = 42
        self.batch_size = 4096
        self.epochs = 100
        self.lr = 0.01


args = Args()

# Verifica i parametri
print(f"Configurazione:")
print(f"  - Model: {args.exp_name}")
print(f"  - Output path: {args.output_path}")
print(f"  - Models path: {args.models_path}")
print(f"  - Seed: {args.seed}")
print(f"  - Batch size: {args.batch_size}")
print(f"  - Epochs: {args.epochs}")
print(f"  - Learning rate: {args.lr}")

In [40]:
print(f"Starting {args.exp_name.upper()} training pipeline with multiple seeds...")

# Setup ambiente
os.makedirs(args.output_path, exist_ok=True)
os.makedirs(args.models_path, exist_ok=True)

In [41]:
# Loading dei set dati
print("Loading pre-prepared data splits...")
train_set, val_set, test_set, X_cols, y_cols = load_splits()

# Preparazione dei dati per tensorflow
X_train, X_val, X_test, y_train, y_val, y_test = prepare_data(
    train_set, val_set, test_set, X_cols, y_cols, args.exp_name
)

In [42]:
# Definisci i seed per eseguire tre training del modello
seeds = [args.seed, args.seed + 1, args.seed + 2]

best_mae = float("inf")
best_model = None
best_results = None
best_seed = None

print(f"\n{'='*60}")
print(f"TRAINING WITH MULTIPLE SEEDS: {seeds}")
print(f"{'='*60}")

In [43]:
# Esegui il training tre volte con tre seed diversi
for i, seed in enumerate(seeds, 1):
    print(f"\n{'='*40}")
    print(f"RUN {i}/3 - SEED {seed}")
    print(f"{'='*40}")

    # Esegui il training
    model, history = train(X_train, y_train, X_val, y_val, args, seed)

    # Valuta il modello
    results = evaluate_model(model, args.exp_name, val_set, X_cols, y_cols)

    # Calcola il MAE
    current_mae = calculate_mae(results)
    print(f"\nSeed {seed} - Patient-based MAE: {current_mae:.4f}")

    # Controlla se questo è il miglior modello finora
    if current_mae < best_mae:
        best_mae = current_mae
        best_model = model
        best_results = results
        best_seed = seed
        print(f"New best model found with seed {seed}!")

    print(f"Current best MAE: {best_mae:.4f} (seed {best_seed})")

In [44]:
# Salva il miglior modello e i risultati
print(f"\n{'='*60}")
print(f"SAVING BEST MODEL")
print(f"{'='*60}")
print(f"Best model achieved with seed {best_seed}")
print(f"Best MAE: {best_mae:.4f}")

# Stampa i risultati finali
print_results(best_results)

In [45]:
# Salva il miglior modello (rename from temporary seed-specific name)
import shutil

temp_model_path = f"{args.models_path}/{args.exp_name}_seed_{best_seed}.weights.h5"
final_model_path = f"{args.models_path}/{args.exp_name}.weights.h5"

if os.path.exists(temp_model_path):
    shutil.move(temp_model_path, final_model_path)
    print(f"Best model saved to: {final_model_path}")

    # Clean up other temporary model files
    for seed in seeds:
        if seed != best_seed:
            temp_path = f"{args.models_path}/{args.exp_name}_seed_{seed}.weights.h5"
            if os.path.exists(temp_path):
                os.remove(temp_path)

# Salva i miglior risultati
output_file = f"{args.output_path}/{args.exp_name}_output.csv"
best_results.to_csv(output_file, index=False)
print(f"Best results saved to: {output_file}")

print(f"\nTraining pipeline completed successfully!")
print(f"Best model trained with seed {best_seed} (MAE: {best_mae:.4f})")